## Imports & IPython Setup

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
from IPython.display import display, Markdown
from google import genai
from google.genai import types
import pandas as pd
import os 

from prompts.finSHIN import fin_shin_taxonomy,fin_shin_mechanics_GAAP,fin_shin_traces_principle,fin_shin_incentives_motivation,fin_shin_shen_clusters,fin_shin_life_arc,fin_shin_quant_sig,fin_shin_early_vs_confirm,fin_shin_csh_flw_grnd_tru,fin_shin_qual_disc_red_flgs,fin_shin_nn_GAAP_metric_abse,fin_shin_acqui_shen,fin_shin_big_bath_cookie_reserv,fin_shin_aud_govern_sig,fin_shin_indus_calib,fin_shin_spectrum,fin_shin_investi_surc,fin_shin_cases,fin_shin_mster_rule,fin_shin_stress_test
from prompts.forensicSUMMARY import eval_prompt,forensic_summary, summary_summary, summary_summary_summary_analyst, summary_summary_summary_fm, forensic_summary_summary, summary_summary_summary_1_analyst, summary_summary_summary_1_FM
from dotenv import load_dotenv
import os
import glob
from pathlib import Path
from pydantic.types import conbytes
import asyncio

load_dotenv()   

InteractiveShell.ast_node_interactivity = "all"
import asyncio
import nest_asyncio
nest_asyncio.apply()


In [ ]:

async def generate_simple_name(uploaded_file):
    prompt = """
    You are a strict financial document classifier.

    Return ONLY JSON:

    {
      "source_name": "<date>_<doc_type>",
      "date": "<fy or quarter>",
      "doc_type": "<type>"
    }

    Rules:
    - date: fy23, fy2023, q1fy24, q2fy23
    - doc_type: [annual_report, concall, transcript, presentation, rating]
    - lowercase only
    - no spaces
    - no explanation
    - if unsure → "unknown" / "other"
    """

    response = await client.aio.models.generate_content(
        model=MODEL_FLASH_25,
        contents=[uploaded_file, prompt],
        config=types.GenerateContentConfig(response_mime_type="application/json")
    )
    try:
        return json.loads(response.text)
    except Exception:
        return None


async def generate_rating_name(uploaded_file):
    prompt = """
    You are a strict rating document classifier.
    Return ONLY JSON:
    {
      "source_name": "<agency>_<date>",
      "date": "<month and year e.g. jan24, mar23>",
      "doc_type": "rating",
      "agency": "<agency name>"
    }
    Rules:
    - agency: [crisil, icra, care, india_ratings, brickwork, acuite]
    - date: jan24, feb23, mar24 (month + 2-digit year)
    - lowercase only, no spaces
    - if unsure → "unknown"
    """
    response = await client.aio.models.generate_content(
        model=MODEL_FLASH_25,
        contents=[uploaded_file, prompt],
        config=types.GenerateContentConfig(response_mime_type="application/json")
    )
    try:
        return json.loads(response.text)
    except Exception:
        return None


def safe_metadata(m):
    if not m or "source_name" not in m:
        return {"source_name": "unknown_other", "doc_type": "other", "date": "unknown"}
    return m


def parse_date_key(date_str: str) -> tuple:
    """
    Converts date strings to sortable tuples.
    fy24    → (2024, 0)
    q2fy24  → (2024, 2)
    fy2023  → (2023, 0)
    unknown → (0, 0)
    """
    date_str = date_str.lower().strip()

    m = re.match(r"q(\d)fy(\d{2,4})$", date_str)
    if m:
        quarter = int(m.group(1))
        year = int(m.group(2))
        if year < 100:
            year += 2000
        return (year, quarter)

    m = re.match(r"fy(\d{2,4})$", date_str)
    if m:
        year = int(m.group(1))
        if year < 100:
            year += 2000
        return (year, 0)

    return (0, 0)


def get_docs(doc_dict, doc_type=None, reverse=True):
    """
    Returns list of full metadata dicts (with file, date, source_name, doc_type).
    Sorted by parsed date, newest first by default.
    """
    items = list(doc_dict.values())
    if doc_type:
        items = [v for v in items if v["doc_type"] == doc_type]
    return sorted(items, key=lambda x: parse_date_key(x["date"]), reverse=reverse)


def batch(lst, size=4):
    return [lst[i:i+size] for i in range(0, len(lst), size)]


## Configuration & Constants

In [ ]:
# ── Constants ─────────────────────────────────────────────────────────────────
company_name = "360 ONE WAM LTD"
API_KEY      = os.getenv("API_KEY")
MODEL_FLASH_3   = "gemini-3-flash-preview"
MODEL_FLASH_25  = "gemini-2.5-flash"
MODEL_PRO       = "gemini-3.1-pro-preview"

# ── Base input directory ───────────────────────────────────────────────────────
BASE_DIR = "/Users/aryamanchandak/Documents/ICICI Prudential/Code/Debt Dashboard/Debt Credit Eval Note/Inputs"

TRANSCRIPTS_DIR = os.path.join(BASE_DIR, "transcripts")
PPT_DIR         = os.path.join(BASE_DIR, "ppt")
AR_DIR          = os.path.join(BASE_DIR, "annual_reports")
CONCALL_DIR     = os.path.join(BASE_DIR, "concalls")
RATINGS_DIR     = os.path.join(BASE_DIR, "ratings")

# ── Gemini client ─────────────────────────────────────────────────────────────
client = genai.Client(api_key=API_KEY)

async def _upload_if_present(path: str):
    """Upload a PDF to Gemini if path is non-empty, else return None."""
    if path and path.strip():
        return await client.aio.files.upload(file=path)
    return None

async def upload_folder(folder_path: str, prefix: str) -> dict:
    """
    Scan a folder for PDFs, sort them, upload all concurrently, and return a dict like:
      { "ar_1": <UploadedFile>, "ar_2": <UploadedFile>, ... }
    Returns empty dict if folder doesn't exist.
    """
    if not os.path.isdir(folder_path):
        print(f"[WARN] Folder not found, skipping: {folder_path}")
        return {}

    pdf_files = sorted(glob.glob(os.path.join(folder_path, "*.pdf")))
    if not pdf_files:
        return {}

    keys = [f"{prefix}_{i}" for i, _ in enumerate(pdf_files, start=1)]
    for key, p in zip(keys, pdf_files):
        print(f"  Queuing {key}: {os.path.basename(p)}")

    uploaded_files = await asyncio.gather(
        *[client.aio.files.upload(file=p) for p in pdf_files]
    )

    return dict(zip(keys, uploaded_files))

# ── Upload all folders concurrently ──────────────────────────────────────────
print("Uploading all folders in parallel...")
ar, tr, ppt, con, rat = await asyncio.gather(
    upload_folder(AR_DIR, "ar"),           # ar["ar_1"], ar["ar_2"], ...
    upload_folder(TRANSCRIPTS_DIR, "tr"),  # tr["tr_1"], ...
    upload_folder(PPT_DIR, "ppt"),         # ppt["ppt_1"], ...
    upload_folder(CONCALL_DIR, "cc"),      # con["cc_1"], ...
    upload_folder(RATINGS_DIR, "rat"),     # rat["rat_1"], ...
)
print("All uploads complete.")


## Helper Functions

In [ ]:

async def generate_section_from_files(
    prompt: str,
    conpr:str,
    *files,
    model: str,
    show_response: bool = False,
    use_search: bool = False,
) -> str:
    """
    Generate a section using a prompt and an arbitrary number of uploaded Gemini file objects.
    Async: uses client.aio for concurrent execution.
    """
    contents = []

    if prompt:
        contents.append(prompt)

    if conpr: 
        contents.append(conpr)

    for file in files:
        if file:
            contents.append(file)

    if not contents:
        raise ValueError("No content provided — prompt, context_prompt, and all files are empty.")

    kwargs = {"model": model, "contents": contents}

    if use_search:
        kwargs["config"] = types.GenerateContentConfig(
            tools=[types.Tool(google_search=types.GoogleSearch())]
        )

    response = await client.aio.models.generate_content(**kwargs)

    if show_response:
        display(Markdown(response.text))

    return response.text

async def summary_agent(
    prompt: str,
    *agent_summaries: str,
    model: str,
    show_response: bool,
    use_search: bool = False,
) -> str:
    """
    Summarises outputs from multiple agents into a single coherent response.
    Async: uses client.aio for concurrent execution.
    """
    contents = []

    if prompt:
        contents.append(prompt)

    for i, summary in enumerate(agent_summaries, start=1):
        if summary and summary.strip():
            contents.append(f"### Agent {i} Output\n{summary.strip()}")

    kwargs = {"model": model, "contents": contents}

    if use_search:
        kwargs["config"] = types.GenerateContentConfig(
            tools=[types.Tool(google_search=types.GoogleSearch())]
        )

    response = await client.aio.models.generate_content(**kwargs)

    if show_response:
        display(Markdown(response.text))

    return response.text


async def run_with_log(coro, label):
    print(f"  [{label}] started")
    result = await coro
    print(f"  [{label}] done ✓")
    return result

### Final Shenanigans Agent

In [ ]:
print ("="*100)
print("FINANCIAL SHENANIGANS ANALYSIS BEGINS")
print ("="*100)

## FIN SHIN Annual report 1

In [ ]:
foundations_framework1, earnings1, cashFLOW1, acquisitionACCOUNTING1, keyMETRICS1 = await asyncio.gather(
    run_with_log(generate_section_from_files(foundations_framework_prompt, ar["ar_1"],model=MODEL_FLASH_25,show_response=True,use_search=False),"foundations_framework"),
    run_with_log(generate_section_from_files(earnings_prompt, ar["ar_1"],model=MODEL_FLASH_25,show_response=True,use_search=False),"earnings"),
    run_with_log(generate_section_from_files(cashFLOW_prompt, ar["ar_1"],model=MODEL_FLASH_25,show_response=True,use_search=False),"cashFLOW"),
    run_with_log(generate_section_from_files(keyMETRICS_prompt, ar["ar_1"],model=MODEL_FLASH_25,show_response=True,use_search=False),"keyMETRICS"),
    run_with_log(generate_section_from_files(acquisitionACCOUNTING_prompt, ar["ar_1"],model=MODEL_FLASH_25,show_response=True,use_search=False),"acquisitionACCOUNTING"),
)

foundations_frameworkI1, earningsI1, cashFLOWI1, acquisitionACCOUNTINGI1, keyMETRICSI1 = await asyncio.gather(
    run_with_log(generate_section_from_files(foundations_framework_promptI, ar["ar_1"],model=MODEL_FLASH_25,show_response=True,use_search=False),"foundations_framework"),
    run_with_log(generate_section_from_files(earnings_promptI, ar["ar_1"],model=MODEL_FLASH_25,show_response=True,use_search=False),"earnings"),
    run_with_log(generate_section_from_files(cashFLOW_promptI, ar["ar_1"],model=MODEL_FLASH_25,show_response=True,use_search=False),"cashFLOW"),
    run_with_log(generate_section_from_files(keyMETRICS_promptI, ar["ar_1"],model=MODEL_FLASH_25,show_response=True,use_search=False),"keyMETRICS"),
    run_with_log(generate_section_from_files(acquisitionACCOUNTING_promptI, ar["ar_1"],model=MODEL_FLASH_25,show_response=True,use_search=False),"acquisitionACCOUNTING"),
)


## Fin Shin Annual Report 2 

In [ ]:
foundations_framework2, earnings2, cashFLOW2, acquisitionACCOUNTING2, keyMETRICS2 = await asyncio.gather(
    run_with_log(generate_section_from_files(foundations_framework_prompt, ar["ar_2"],model=MODEL_FLASH_25,show_response=True,use_search=False),"foundations_framework"),
    run_with_log(generate_section_from_files(earnings_prompt, ar["ar_2"],model=MODEL_FLASH_25,show_response=True,use_search=False),"earnings"),
    run_with_log(generate_section_from_files(cashFLOW_prompt, ar["ar_2"],model=MODEL_FLASH_25,show_response=True,use_search=False),"cashFLOW"),
    run_with_log(generate_section_from_files(keyMETRICS_prompt, ar["ar_2"],model=MODEL_FLASH_25,show_response=True,use_search=False),"keyMETRICS"),
    run_with_log(generate_section_from_files(acquisitionACCOUNTING_prompt, ar["ar_2"],model=MODEL_FLASH_25,show_response=True,use_search=False),"acquisitionACCOUNTING"),
)

foundations_frameworkI2, earningsI2, cashFLOWI2, acquisitionACCOUNTINGI2, keyMETRICSI2 = await asyncio.gather(
    run_with_log(generate_section_from_files(foundations_framework_promptI, ar["ar_2"],model=MODEL_FLASH_25,show_response=True,use_search=False),"foundations_framework"),
    run_with_log(generate_section_from_files(earnings_promptI, ar["ar_2"],model=MODEL_FLASH_25,show_response=True,use_search=False),"earnings"),
    run_with_log(generate_section_from_files(cashFLOW_promptI, ar["ar_2"],model=MODEL_FLASH_25,show_response=True,use_search=False),"cashFLOW"),
    run_with_log(generate_section_from_files(keyMETRICS_promptI, ar["ar_2"],model=MODEL_FLASH_25,show_response=True,use_search=False),"keyMETRICS"),
    run_with_log(generate_section_from_files(acquisitionACCOUNTING_promptI, ar["ar_2"],model=MODEL_FLASH_25,show_response=True,use_search=False),"acquisitionACCOUNTING"),
)


## Fin Shin Annual Report 3

In [ ]:
foundations_framework3, earnings3, cashFLOW3, acquisitionACCOUNTING3, keyMETRICS3 = await asyncio.gather(
    run_with_log(generate_section_from_files(foundations_framework_prompt, ar["ar_3"],model=MODEL_FLASH_25,show_response=True,use_search=False),"foundations_framework"),
    run_with_log(generate_section_from_files(earnings_prompt, ar["ar_3"],model=MODEL_FLASH_25,show_response=True,use_search=False),"earnings"),
    run_with_log(generate_section_from_files(cashFLOW_prompt, ar["ar_3"],model=MODEL_FLASH_25,show_response=True,use_search=False),"cashFLOW"),
    run_with_log(generate_section_from_files(keyMETRICS_prompt, ar["ar_3"],model=MODEL_FLASH_25,show_response=True,use_search=False),"keyMETRICS"),
    run_with_log(generate_section_from_files(acquisitionACCOUNTING_prompt, ar["ar_3"],model=MODEL_FLASH_25,show_response=True,use_search=False),"acquisitionACCOUNTING"),
)

foundations_frameworkI3, earningsI3, cashFLOWI3, acquisitionACCOUNTINGI3, keyMETRICSI3 = await asyncio.gather(
    run_with_log(generate_section_from_files(foundations_framework_promptI, ar["ar_3"],model=MODEL_FLASH_25,show_response=True,use_search=False),"foundations_framework"),
    run_with_log(generate_section_from_files(earnings_promptI, ar["ar_3"],model=MODEL_FLASH_25,show_response=True,use_search=False),"earnings"),
    run_with_log(generate_section_from_files(cashFLOW_promptI, ar["ar_3"],model=MODEL_FLASH_25,show_response=True,use_search=False),"cashFLOW"),
    run_with_log(generate_section_from_files(keyMETRICS_promptI, ar["ar_3"],model=MODEL_FLASH_25,show_response=True,use_search=False),"keyMETRICS"),
    run_with_log(generate_section_from_files(acquisitionACCOUNTING_promptI, ar["ar_3"],model=MODEL_FLASH_25,show_response=True,use_search=False),"acquisitionACCOUNTING"),
)


## FIN SHIN Annual report 4

In [ ]:
foundations_framework4, earnings4, cashFLOW4, acquisitionACCOUNTING4, keyMETRICS4 = await asyncio.gather(
    run_with_log(generate_section_from_files(foundations_framework_prompt, ar["ar_3"],model=MODEL_FLASH_25,show_response=True,use_search=False),"foundations_framework"),
    run_with_log(generate_section_from_files(earnings_prompt, ar["ar_3"],model=MODEL_FLASH_25,show_response=True,use_search=False),"earnings"),
    run_with_log(generate_section_from_files(cashFLOW_prompt, ar["ar_3"],model=MODEL_FLASH_25,show_response=True,use_search=False),"cashFLOW"),
    run_with_log(generate_section_from_files(keyMETRICS_prompt, ar["ar_3"],model=MODEL_FLASH_25,show_response=True,use_search=False),"keyMETRICS"),
    run_with_log(generate_section_from_files(acquisitionACCOUNTING_prompt, ar["ar_3"],model=MODEL_FLASH_25,show_response=True,use_search=False),"acquisitionACCOUNTING"),
)

foundations_frameworkI4, earningsI4, cashFLOWI4, acquisitionACCOUNTINGI4, keyMETRICSI4 = await asyncio.gather(
    run_with_log(generate_section_from_files(foundations_framework_promptI, ar["ar_3"],model=MODEL_FLASH_25,show_response=True,use_search=False),"foundations_framework"),
    run_with_log(generate_section_from_files(earnings_promptI, ar["ar_3"],model=MODEL_FLASH_25,show_response=True,use_search=False),"earnings"),
    run_with_log(generate_section_from_files(cashFLOW_promptI, ar["ar_3"],model=MODEL_FLASH_25,show_response=True,use_search=False),"cashFLOW"),
    run_with_log(generate_section_from_files(keyMETRICS_promptI, ar["ar_3"],model=MODEL_FLASH_25,show_response=True,use_search=False),"keyMETRICS"),
    run_with_log(generate_section_from_files(acquisitionACCOUNTING_promptI, ar["ar_3"],model=MODEL_FLASH_25,show_response=True,use_search=False),"acquisitionACCOUNTING"),
)


## Fin Shin Summary 1 AR

In [ ]:
summary_fm, summary_analyst = await asyncio.gather(
    summary_agent(summary_summary_summary_1_FM,foundations_framework1,earnings1,cashFLOW1,acquisitionACCOUNTING1,keyMETRICS1, model = MODEL_FLASH_3, show_response=True, use_search=False),
    summary_agent(summary_summary_summary_1_analyst, foundations_framework1,earnings1,cashFLOW1,acquisitionACCOUNTING1,keyMETRICS1, model = MODEL_FLASH_3, show_response=True, use_search=False),
)


## FIN SHIN Summary 4 AR

In [ ]:
foundations_framework_summary, earnings_summary, cashFLOW_summary, acquisitionACCOUNTING_summary, keyMETRICS_summary = await asyncio.gather(
    summary_agent(forensic_summary, foundations_framework1, foundations_framework2,foundations_framework3,foundations_framework4, model=MODEL_FLASH_3, show_response=True, use_search=False),
    summary_agent(forensic_summary, earnings1,earnings2,earnings3,earnings4,  model=MODEL_FLASH_3, show_response=True, use_search=False),
    summary_agent(forensic_summary, cashFLOW1,cashFLOW2,cashFLOW3,cashFLOW4,  model=MODEL_FLASH_3, show_response=True, use_search=False),
    summary_agent(forensic_summary, acquisitionACCOUNTING1,acquisitionACCOUNTING2,acquisitionACCOUNTING3,acquisitionACCOUNTING4,  model=MODEL_FLASH_3, show_response=True, use_search=False),
    summary_agent(forensic_summary, keyMETRICS1,keyMETRICS2,keyMETRICS3,keyMETRICS4,  model=MODEL_FLASH_3, show_response=True, use_search=False),
)

summary_fm,summary_analyst = await asyncio.gather(
    summary_agent(summary_summary_summary_1_FM,foundations_framework_summary,earnings_summary,cashFLOW_summary,acquisitionACCOUNTING_summary,keyMETRICS_summary, model = MODEL_FLASH_3, show_response=True, use_search=False),
    summary_agent(summary_summary_summary_1_analyst, foundations_framework_summary,earnings_summary,cashFLOW_summary,acquisitionACCOUNTING_summary,keyMETRICS_summary, model = MODEL_FLASH_3, show_response=True, use_search=False),
)


## Fin Shin Summary I Version

In [ ]:
foundations_framework_summary, earnings_summary, cashFLOW_summary, acquisitionACCOUNTING_summary, keyMETRICS_summary = await asyncio.gather(
    summary_agent(forensic_summary, foundations_frameworkI1, foundations_frameworkI2,foundations_frameworkI3,foundations_frameworkI4, model=MODEL_FLASH_3, show_response=True, use_search=False),
    summary_agent(forensic_summary, earningsI1,earningsI2,earningsI3,earningsI4,  model=MODEL_FLASH_3, show_response=True, use_search=False),
    summary_agent(forensic_summary, cashFLOWI1,cashFLOWI2,cashFLOW3,cashFLOW4,  model=MODEL_FLASH_3, show_response=True, use_search=False),
    summary_agent(forensic_summary, acquisitionACCOUNTING1,acquisitionACCOUNTING2,acquisitionACCOUNTING3,acquisitionACCOUNTING4,  model=MODEL_FLASH_3, show_response=True, use_search=False),
    summary_agent(forensic_summary, keyMETRICS1,keyMETRICS2,keyMETRICS3,keyMETRICS4,  model=MODEL_FLASH_3, show_response=True, use_search=False),
)

summary_fm,summary_analyst = await asyncio.gather(
    summary_agent(summary_summary_summary_1_FM,foundations_framework_summary,earnings_summary,cashFLOW_summary,acquisitionACCOUNTING_summary,keyMETRICS_summary, model = MODEL_FLASH_3, show_response=True, use_search=False),
    summary_agent(summary_summary_summary_1_analyst, foundations_framework_summary,earnings_summary,cashFLOW_summary,acquisitionACCOUNTING_summary,keyMETRICS_summary, model = MODEL_FLASH_3, show_response=True, use_search=False),
)
